In [1]:
import gcsfs
import pandas as pd

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

In [2]:
EXISTING_GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
existing_annual = pd.read_parquet(
    f"{EXISTING_GCS}annual_ridership_report_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [3]:
def merge_new_df_with_crosswalk(
    filename: str = "annual"
):
    """
    """
    crosswalk = pd.read_parquet(
        f"{GCS_FILE_PATH}crosswalk.parquet", 
        filesystem=gcsfs.GCSFileSystem(),
    ).rename(columns = {"ntd_id_2022": "ntd_id"}).drop_duplicates()

    df = pd.read_parquet(
        f"{GCS_FILE_PATH}{filename}.parquet",
        filesystem=gcsfs.GCSFileSystem(),
    ).merge(
        crosswalk,
        on = "ntd_id",
        how = "left"
    )

    return df

In [4]:
df = merge_new_df_with_crosswalk("annual")

In [5]:
def counts_by_rtpa(
    df: pd.DataFrame,
    group_cols: list
) -> pd.DataFrame:
    """
    Use this to read in existing df vs new annual/monthly df
    and do groupby by rtpa_name or rtpa_name_split,
    and see how counts look overall.
    """
    df2 = (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            total_upt=("upt", "sum"),
            n_agencies=("source_agency", "nunique"),
            agencies=pd.NamedAgg(column="source_agency", aggfunc=lambda x: list(x)),
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by=group_cols + ["total_upt"], ascending=False)
        .reset_index(drop=True)
    )

    return df2

In [6]:
df2 = counts_by_rtpa(df.rename(columns = {"unlinked_passenger_trips": "upt"}), ["rtpa_name"])

In [7]:
existing_df2 = counts_by_rtpa(existing_annual, ["rtpa_name"])

In [8]:
compare_df = pd.merge(
    existing_df2, 
    df2,
    on = "rtpa_name",
    how = "outer",
    indicator=True
).astype({
    c: "Int64" 
    for c in ["total_upt_x", "total_upt_y", "n_agencies_x", "n_agencies_y"]}
)

In [9]:
compare_df = compare_df.assign(
    missing_agency = compare_df.apply(
        lambda x:
        list(set([c for c in x.agencies_x if c not in x.agencies_y])) if x._merge=="both"
        else [], 
        axis=1)
)

In [10]:
# bridge is undercounting agencies, which results in fewer upt
results = compare_df[(compare_df._merge=="both") & 
    (compare_df.total_upt_x != compare_df.total_upt_y)]

In [12]:
results

,rtpa_name,total_upt_x,n_agencies_x,agencies_x,total_upt_y,n_agencies_y,agencies_y,_merge,missing_agency
1,Del Norte Local Transportation Commission,84385,2,"[Yurok Tribe - Transportation Department, Yuro...",84077,1,"[Yurok Tribe - Transportation Department, Yuro...",both,[Elk Valley Rancheria (EVR)]
7,Kings County Association of Governments,28416124,2,[Kings County Area Public Transit Agency (KART...,4065937,1,[Kings County Area Public Transit Agency (KART...,both,[California Vanpool Authority (CVA)]
11,Madera County Transportation Commission,834898,3,"[City of Madera (MAX/DAR) - Transit, City of M...",824512,2,"[City of Madera (MAX/DAR) - Transit, City of M...",both,[North Fork Rancheria of Mono Indians of Calif...
13,Metropolitan Transportation Commission,2331534369,23,[San Francisco Bay Area Rapid Transit District...,2308006739,20,[San Francisco Bay Area Rapid Transit District...,both,[County of Sonoma (SCT) - Department of Public...
17,Sacramento Area Council of Governments,142667186,10,"[Sacramento Regional Transit District, Sacrame...",141160514,6,"[Sacramento Regional Transit District, Sacrame...",both,"[Attentive Transportation LLC, City of Davis (..."
19,San Diego Association of Governments,546401079,4,"[San Diego Metropolitan Transit System (MTS), ...",54570070,1,"[North County Transit District (NCTD), North C...",both,"[City of Atascadero - Public Works, San Diego ..."
20,San Joaquin Council of Governments,30093684,7,"[San Joaquin Regional Transit District (RTD), ...",26534036,6,"[San Joaquin Regional Transit District (RTD), ...",both,[San Joaquin Council (SJCOG)]
21,San Luis Obispo Council of Governments,10236333,3,"[City of San Luis Obispo - Public Works, City ...",10159067,2,"[City of San Luis Obispo - Public Works, City ...",both,[San Luis Obispo Council of Governments (SLOCOG)]
22,Santa Barbara County Association of Governments,38826821,5,[Santa Barbara Metropolitan Transit District (...,38387059,4,[Santa Barbara Metropolitan Transit District (...,both,[Easy Lift Transportation]
26,Stanislaus Council of Governments,18476060,5,"[City of Modesto (MAX), City of Modesto (MAX),...",8924915,2,"[City of Turlock - Transit, City of Turlock - ...",both,"[Stanislaus County (StaRT), Stanislaus Council..."


In [11]:
for rtpa in results.rtpa_name.unique():
    print(rtpa)
    print(results[results.rtpa_name==rtpa].reset_index(drop=True).missing_agency.iloc[0])
    print("******************************************************************")

Del Norte Local Transportation Commission
['Elk Valley Rancheria (EVR)']
******************************************************************
Kings County Association of Governments
['California Vanpool Authority (CVA)']
******************************************************************
Madera County Transportation Commission
['North Fork Rancheria of Mono Indians of California (NFR) - Administration Department']
******************************************************************
Metropolitan Transportation Commission
['County of Sonoma (SCT) - Department of Public Infrastructure - Transit Division', 'Metropolitan Transportation Commission (MTC) - Field Operations and Asset Management', 'San Francisco Bay Area Water Emergency Transportation Authority (WETA)']
******************************************************************
Sacramento Area Council of Governments
['Attentive Transportation LLC', 'City of Davis (DCT) - Transit/Parks and Community Services', 'Paratransit, Inc.', 'City of Fo